# CN7: 검사 전 불량 예측, 실패 조건 분석, 검사 우선순위
원본 노트북의 센서 + Side_RH + 직전 5 Cycle 평균과 LightGBM 설정을 사용합니다.
주의: 이 데이터의 불량 17건은 모두 첫날에 있습니다. 이후 날짜의 재현율은 측정할 수 없습니다.



In [1]:
from pathlib import Path
import warnings
import joblib
import numpy as np
import pandas as pd
from lightgbm import LGBMClassifier
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import average_precision_score, precision_score, recall_score, fbeta_score

DATA_PATH = Path(r"C:\Users\Administrator\Desktop\kamp\1. 사출성형기 AI 데이터셋\1. 사출성형기 AI 데이터셋\moldset_labeled_cn7_되돌림.csv")
OUTPUT_DIR = Path.cwd() / "cn7_results"
OUTPUT_DIR.mkdir(exist_ok=True)
DEVELOPMENT_END = pd.Timestamp("2020-10-16 23:59:59")
INSPECTION_CAPACITY_RATE = 0.02  # 날짜별 검사 가능 수량: 생산량의 2%, 현장 여건에 맞게 변경
SENSOR_AVAILABILITY_CONFIRMED = False  # 모든 입력값이 최종 품질검사 전에 확정되면 True로 변경

PARAMS = dict(
    n_estimators=200, learning_rate=0.10, num_leaves=15,
    max_depth=7, min_child_samples=10, subsample=1.0,
    colsample_bytree=0.70, reg_alpha=0.0, reg_lambda=0.0,
    class_weight="balanced", random_state=42, verbose=-1,
)



## 1. 데이터 확인과 과거 정보만 사용하는 특성 생성
현재 Cycle의 센서값과 직전 5 Cycle의 평균만 사용합니다. 라벨은 특성 생성에 사용하지 않습니다.



In [2]:
raw = pd.read_csv(DATA_PATH)
raw["TimeStamp"] = pd.to_datetime(raw["TimeStamp"], errors="raise")
assert raw["PassOrFail"].isin([0, 1]).all()
assert raw["RH_LH"].isin(["LH", "RH"]).all()
assert not raw.duplicated(["TimeStamp", "RH_LH"]).any()

EXCLUDE = {"original_index", "PassOrFail", "Clamp_Open_Position", "PART_NAME", "RH_LH", "TimeStamp"}
SENSOR_COLS = [c for c in raw.columns if c not in EXCLUDE]
assert len(SENSOR_COLS) == 23, SENSOR_COLS
assert raw[SENSOR_COLS].notna().all().all()
assert all(pd.api.types.is_numeric_dtype(raw[c]) for c in SENSOR_COLS)
FEATURES = SENSOR_COLS + ["Side_RH"] + [f"{c}_roll5_mean" for c in SENSOR_COLS]

def make_features(frame):
    """라벨이 없는 새 생산 데이터에도 동일하게 적용할 수 있는 전처리."""
    x = frame.copy().sort_values(["TimeStamp", "RH_LH"]).reset_index(drop=True)
    x["Side_RH"] = x["RH_LH"].eq("RH").astype(int)
    cycle = x.drop_duplicates("TimeStamp").sort_values("TimeStamp").copy()
    cycle["session_id"] = cycle["TimeStamp"].diff().dt.total_seconds().gt(300).cumsum()
    cycle["history_cycles"] = cycle.groupby("session_id").cumcount().clip(upper=5)
    for col in SENSOR_COLS:
        cycle[f"{col}_roll5_mean"] = (
            cycle.groupby("session_id")[col]
            .transform(lambda s: s.shift(1).rolling(5, min_periods=1).mean())
            .fillna(0)
        )
    roll = [f"{c}_roll5_mean" for c in SENSOR_COLS]
    return x.merge(cycle[["TimeStamp", "session_id", "history_cycles"] + roll],
                   on="TimeStamp", how="left", validate="many_to_one")

data = make_features(raw)
development = data.loc[data.TimeStamp <= DEVELOPMENT_END].reset_index(drop=True)
future = data.loc[data.TimeStamp > DEVELOPMENT_END].reset_index(drop=True)
print("전체 / 개발 / 이후 생산:", len(data), len(development), len(future))
print("불량 건수:", data.PassOrFail.sum(), development.PassOrFail.sum(), future.PassOrFail.sum())
print("검사 전 센서 확보 확인:", SENSOR_AVAILABILITY_CONFIRMED)
if not SENSOR_AVAILABILITY_CONFIRMED:
    warnings.warn("공정 담당자가 23개 센서의 검사 전 확보 시점을 확인해야 운영할 수 있습니다.")



전체 / 개발 / 이후 생산: 1211 375 836
불량 건수: 17 17 0
검사 전 센서 확보 확인: False


C:\Users\Administrator\AppData\Local\Temp\ipykernel_46948\2048337117.py:38: UserWarning: 공정 담당자가 23개 센서의 검사 전 확보 시점을 확인해야 운영할 수 있습니다.
  warnings.warn("공정 담당자가 23개 센서의 검사 전 확보 시점을 확인해야 운영할 수 있습니다.")


## 2. 그룹 교차검증과 임계값 선택
동일 Cycle의 LH/RH는 같은 Fold에 둡니다. 각 바깥 Fold의 임계값은 해당 Fold의
학습 데이터에서만 선택하여 검증 데이터에 적용합니다.



In [3]:
def new_model():
    return LGBMClassifier(**PARAMS)

def group_oof(frame, n_splits=3, seed=42):
    cv = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    score = np.full(len(frame), np.nan)
    y = frame.PassOrFail.to_numpy()
    for tr, va in cv.split(frame[FEATURES], y, groups=frame.TimeStamp):
        if len(np.unique(y[tr])) < 2:
            raise ValueError("학습 Fold에 정상과 불량이 모두 필요합니다.")
        model = new_model().fit(frame.loc[tr, FEATURES], y[tr])
        score[va] = model.predict_proba(frame.loc[va, FEATURES])[:, 1]
    assert np.isfinite(score).all()
    return score

def choose_threshold(y, score):
    candidates = np.arange(0.005, 0.501, 0.005)
    # 동점이면 더 높은 임계값을 선택하여 검사 수량을 줄입니다.
    return max(candidates, key=lambda t: (fbeta_score(y, score >= t, beta=2, zero_division=0), t))

outer = StratifiedGroupKFold(n_splits=4, shuffle=True, random_state=42)
outer_score = np.full(len(development), np.nan)
outer_pred = np.zeros(len(development), dtype=int)
outer_threshold = np.full(len(development), np.nan)
outer_fold = np.zeros(len(development), dtype=int)
y_dev = development.PassOrFail.to_numpy()
for fold, (tr, va) in enumerate(outer.split(development[FEATURES], y_dev, groups=development.TimeStamp), 1):
    train = development.iloc[tr].reset_index(drop=True)
    inner_score = group_oof(train, n_splits=3, seed=42 + fold)
    threshold = choose_threshold(train.PassOrFail.to_numpy(), inner_score)
    model = new_model().fit(train[FEATURES], train.PassOrFail)
    outer_score[va] = model.predict_proba(development.loc[va, FEATURES])[:, 1]
    outer_pred[va] = (outer_score[va] >= threshold).astype(int)
    outer_threshold[va] = threshold
    outer_fold[va] = fold
    print(f"Fold {fold}: 학습 불량={train.PassOrFail.sum()}, 검증 불량={y_dev[va].sum()}, 임계값={threshold:.3f}")

assert np.isfinite(outer_score).all()
print("교차검증 PR-AUC:", round(average_precision_score(y_dev, outer_score), 4))
print("교차검증 Precision:", round(precision_score(y_dev, outer_pred, zero_division=0), 4))
print("교차검증 Recall:", round(recall_score(y_dev, outer_pred, zero_division=0), 4))
print("교차검증 F2:", round(fbeta_score(y_dev, outer_pred, beta=2, zero_division=0), 4))



Fold 1: 학습 불량=13, 검증 불량=4, 임계값=0.080
Fold 2: 학습 불량=13, 검증 불량=4, 임계값=0.015
Fold 3: 학습 불량=12, 검증 불량=5, 임계값=0.240
Fold 4: 학습 불량=13, 검증 불량=4, 임계값=0.500
교차검증 PR-AUC: 0.7722
교차검증 Precision: 0.6667
교차검증 Recall: 0.5882
교차검증 F2: 0.6024


## 3. 맞는 조건과 실패 조건
FN은 놓친 불량, FP는 불필요한 검사 후보입니다. 표본 수를 함께 표시합니다.
이 데이터는 불량이 짧은 시간에 몰려 있으므로 아래 조건은 탐색 결과입니다.



In [4]:
review = development[["TimeStamp", "RH_LH", "session_id", "history_cycles", "PassOrFail"]].copy()
review["risk_score"] = outer_score
review["prediction"] = outer_pred
review["threshold"] = outer_threshold
review["fold"] = outer_fold
review["error_type"] = np.select(
    [(review.PassOrFail == 1) & (review.prediction == 0),
     (review.PassOrFail == 0) & (review.prediction == 1)],
    ["FN_missed_defect", "FP_extra_inspection"], default="correct"
)
review["history_band"] = pd.cut(review.history_cycles, [-1, 0, 4, 5],
                                labels=["첫 Cycle", "직전 1~4개", "직전 5개"])

def condition_table(frame, col):
    rows = []
    for value, part in frame.groupby(col, observed=True, dropna=False):
        tp = int(((part.PassOrFail == 1) & (part.prediction == 1)).sum())
        fn = int(((part.PassOrFail == 1) & (part.prediction == 0)).sum())
        fp = int(((part.PassOrFail == 0) & (part.prediction == 1)).sum())
        n_bad = tp + fn
        rows.append({"condition": col, "value": str(value), "count": len(part),
                     "defects": n_bad, "TP": tp, "FN": fn, "FP": fp,
                     "recall": tp / n_bad if n_bad else np.nan,
                     "false_alarm_rate": fp / (len(part) - n_bad) if len(part) > n_bad else np.nan})
    return pd.DataFrame(rows)

# 원본 노트북의 SHAP 결과에 있던 주요 공정 변수도 구간별로 확인합니다.
for col in ["Mold_Temperature_3_roll5_mean", "Mold_Temperature_4_roll5_mean", "Plasticizing_Position"]:
    review[col] = development[col]
    review[col + "_band"] = pd.qcut(review[col], q=4, duplicates="drop").astype(str)

conditions = pd.concat([
    condition_table(review, col) for col in
    ["RH_LH", "history_band", "Mold_Temperature_3_roll5_mean_band",
     "Mold_Temperature_4_roll5_mean_band", "Plasticizing_Position_band"]
], ignore_index=True)
print(conditions.to_string(index=False))
print("\n놓친 불량과 추가 검사 후보:")
print(review.loc[review.error_type != "correct"].to_string(index=False))
conditions.to_csv(OUTPUT_DIR / "conditions.csv", index=False, encoding="utf-8-sig")
review.to_csv(OUTPUT_DIR / "cv_errors.csv", index=False, encoding="utf-8-sig")



                         condition                       value  count  defects  TP  FN  FP   recall  false_alarm_rate
                             RH_LH                          LH    187        4   3   1   2 0.750000          0.010929
                             RH_LH                          RH    188       13   7   6   3 0.538462          0.017143
                      history_band                     첫 Cycle      1        0   0   0   1      NaN          1.000000
                      history_band                     직전 1~4개      8        0   0   0   1      NaN          0.125000
                      history_band                       직전 5개    366       17  10   7   3 0.588235          0.008596
Mold_Temperature_3_roll5_mean_band             (-0.001, 1.152]    119        3   2   1   3 0.666667          0.025862
Mold_Temperature_3_roll5_mean_band              (1.152, 1.166]     72        5   5   0   0 1.000000          0.000000
Mold_Temperature_3_roll5_mean_band              (1.166, 

## 4. 이후 생산 날짜 검증
이후 데이터는 불량이 0건이므로 Recall/PR-AUC를 계산하지 않습니다.
검사 후보 수와 정상 제품의 오탐 비율만 확인합니다.



In [5]:
final_oof = group_oof(development, n_splits=4, seed=100)
operation_threshold = float(choose_threshold(y_dev, final_oof))
final_model = new_model().fit(development[FEATURES], development.PassOrFail)
future = future.copy()
future["risk_score"] = final_model.predict_proba(future[FEATURES])[:, 1]
future["inspection_flag"] = future.risk_score >= operation_threshold
future["date"] = future.TimeStamp.dt.date
future_summary = future.groupby("date").agg(
    produced=("PassOrFail", "size"), defects=("PassOrFail", "sum"),
    inspection_candidates=("inspection_flag", "sum"),
    median_score=("risk_score", "median"), max_score=("risk_score", "max")
).reset_index()
future_summary["false_alarm_rate"] = np.where(
    future_summary.defects.eq(0),
    future_summary.inspection_candidates / future_summary.produced,
    np.nan
)
print("운영 임계값:", operation_threshold)
print(future_summary.to_string(index=False))
future_summary.to_csv(OUTPUT_DIR / "future_validation.csv", index=False, encoding="utf-8-sig")



운영 임계값: 0.02
      date  produced  defects  inspection_candidates  median_score  max_score  false_alarm_rate
2020-10-19        72        0                     48      0.307376   0.997662          0.666667
2020-10-20       764        0                    362      0.016083   0.999973          0.473822


## 5. 불량 위험 점수로 검사 순서 결정
점수는 순위용입니다. class_weight를 썼으므로 0.20을 불량 확률 20%로 해석하지 않습니다.
날짜별 검사 가능 수량만큼 높은 점수의 제품을 우선 배정합니다.



In [6]:
def make_priority_table(scored, capacity_rate=INSPECTION_CAPACITY_RATE):
    out = scored[["TimeStamp", "RH_LH", "risk_score"]].copy()
    out["date"] = out.TimeStamp.dt.date
    out = out.sort_values(["date", "risk_score", "TimeStamp", "RH_LH"],
                          ascending=[True, False, True, True]).reset_index(drop=True)
    out["priority_rank"] = out.groupby("date").cumcount() + 1
    out["capacity"] = out.groupby("date")["date"].transform("size").mul(capacity_rate).apply(np.ceil).astype(int)
    out["inspect_first"] = out.priority_rank <= out.capacity
    out["above_threshold"] = out.risk_score >= operation_threshold
    return out

priority = make_priority_table(future)
print(priority.head(20).to_string(index=False))
priority.to_csv(OUTPUT_DIR / "inspection_priority.csv", index=False, encoding="utf-8-sig")

# 실제 검사 전 새 데이터에 적용할 함수. history에는 새 배치보다 이전 Cycle을 넣습니다.
def score_new_batch(history, new_batch, model=final_model, capacity_rate=INSPECTION_CAPACITY_RATE):
    need = ["TimeStamp", "RH_LH"] + SENSOR_COLS
    hist = history[need].copy()
    batch = new_batch[need].copy()
    hist["TimeStamp"] = pd.to_datetime(hist.TimeStamp)
    batch["TimeStamp"] = pd.to_datetime(batch.TimeStamp)
    if len(hist) and hist.TimeStamp.max() >= batch.TimeStamp.min():
        raise ValueError("history는 새 배치의 첫 Cycle보다 이전이어야 합니다.")
    if batch.empty or batch[need].isna().any().any():
        raise ValueError("새 배치가 비었거나 검사 전 입력값에 결측치가 있습니다.")
    hist["_new"] = False
    batch["_new"] = True
    features = make_features(pd.concat([hist, batch], ignore_index=True))
    current = features.loc[features._new].copy()
    current["risk_score"] = model.predict_proba(current[FEATURES])[:, 1]
    return make_priority_table(current, capacity_rate)

artifact = {
    "model": final_model, "sensor_cols": SENSOR_COLS, "features": FEATURES,
    "threshold": operation_threshold, "capacity_rate": INSPECTION_CAPACITY_RATE,
    "development_end": str(DEVELOPMENT_END),
    "sensor_availability_confirmed": SENSOR_AVAILABILITY_CONFIRMED,
}
joblib.dump(artifact, OUTPUT_DIR / "cn7_model.joblib")
print("저장 위치:", OUTPUT_DIR)
print("운영 전 확인: 센서 확정 시점, 이후 기간의 실제 불량 사례, 검사 가능 수량")


          TimeStamp RH_LH  risk_score       date  priority_rank  capacity  inspect_first  above_threshold
2020-10-19 23:57:30    RH    0.997662 2020-10-19              1         2           True             True
2020-10-19 23:58:30    RH    0.995851 2020-10-19              2         2           True             True
2020-10-19 23:29:45    RH    0.995554 2020-10-19              3         2          False             True
2020-10-19 23:59:28    RH    0.995510 2020-10-19              4         2          False             True
2020-10-19 23:48:34    RH    0.992302 2020-10-19              5         2          False             True
2020-10-19 23:48:34    LH    0.992278 2020-10-19              6         2          False             True
2020-10-19 23:53:33    RH    0.991689 2020-10-19              7         2          False             True
2020-10-19 23:45:36    RH    0.986820 2020-10-19              8         2          False             True
2020-10-19 23:28:45    RH    0.985747 2020-10-